In [4]:
import onnxruntime as ort

In [1]:
# Download the reference model, its external weights and the sample image
PREFIX = "https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle"
!curl.exe -fsSL -o hair_classifier_v1.onnx.data "{PREFIX}/hair_classifier_v1.onnx.data"
!curl.exe -fsSL -o hair_classifier_v1.onnx "{PREFIX}/hair_classifier_v1.onnx"
!curl.exe -fsSL -o sample.jpeg "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"

In [2]:
# Verify the SHA-256 checksums against asset_manifest.json
import hashlib, json

manifest = json.load(open("asset_manifest.json"))
files = {"model": "hair_classifier_v1.onnx",
         "external_data": "hair_classifier_v1.onnx.data",
         "sample_image": "sample.jpeg"}

for key, name in files.items():
    actual = hashlib.sha256(open(name, "rb").read()).hexdigest()     # hash of the downloaded file
    expected = manifest[key]["sha256"]                               # hash from the manifest
    print(f"{name}\n  file:     {actual}\n  manifest: {expected}\n  match:    {actual == expected}")

hair_classifier_v1.onnx
  file:     2b1adcb51745b73609ac7efebf3b6199483a931ed6dfe35ab925978f4357c2d8
  manifest: 2b1adcb51745b73609ac7efebf3b6199483a931ed6dfe35ab925978f4357c2d8
  match:    True
hair_classifier_v1.onnx.data
  file:     6ba88582d6098a535f918d9a56ad23800f919ad7bd5aad67e5b2bb3e289e5a7e
  manifest: 6ba88582d6098a535f918d9a56ad23800f919ad7bd5aad67e5b2bb3e289e5a7e
  match:    True
sample.jpeg
  file:     6b3ae4003b5b6d54b51ee4d5fc5f183126bb1be0bf7066e5a846785fdcbfd2f5
  manifest: 6b3ae4003b5b6d54b51ee4d5fc5f183126bb1be0bf7066e5a846785fdcbfd2f5
  match:    True


In [3]:
# Install the pinned dependencies for this homework (already done)
# %pip install -r requirements.txt

##### Question 1 — look at the ONNX graph

In [5]:
# Load the model and inspect its input and output nodes
session = ort.InferenceSession("hair_classifier_v1.onnx")   # also loads the .onnx.data file next to it

for i in session.get_inputs():
    print("Input: ", i.name, i.shape, i.type)
for o in session.get_outputs():
    print("Output:", o.name, o.shape, o.type)

Input:  input ['s77', 3, 200, 200] tensor(float)
Output: output ['s77', 1] tensor(float)


**Explanation:** The model has one input `input` with shape `[batch, 3, 200, 200]` (`s77` is the
dynamic batch dimension) and one output `output` with shape `[batch, 1]`:
the probability of straight hair.